In [19]:
%pip install pandas numpy scikit-learn matplotlib seaborn streamlit joblib


Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

import streamlit as st
import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [21]:
import pandas as pd
import requests
import zipfile
import io

# Download the UCI Bank Marketing dataset
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

response = requests.get(url)
response.raise_for_status()

# Open the outer ZIP file
outer_zip = zipfile.ZipFile(io.BytesIO(response.content))

# Extract and open the inner bank.zip
inner_zip = zipfile.ZipFile(io.BytesIO(outer_zip.read("bank.zip")))

# Read bank.csv from the inner ZIP
df = pd.read_csv(
    io.BytesIO(inner_zip.read("bank.csv")),
    sep=";"
)

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [22]:
# Separate features and target
X = df.drop("y", axis=1)
y = df["y"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Total records:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of features:", X.shape[1])

print("\nTarget distribution:")
print(y.value_counts())

Total records: 4521
Training records: 3616
Testing records: 905
Number of features: 16

Target distribution:
y
no     4000
yes     521
Name: count, dtype: int64


In [23]:
import pandas as pd

# Load dataset using semicolon delimiter
df = pd.read_csv("bank.csv", sep=";")

# Remove any extra spaces from column names
df.columns = df.columns.str.strip()

print("Column names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

First 5 rows:
   age          job  marital  education default  balance housing loan  \
0   30   unemployed  married    primary      no     1787      no   no   
1   33     services  married  secondary      no     4789     yes  yes   
2   35   management   single   tertiary      no     1350     yes   no   
3   30   management  married   tertiary      no     1476     yes  yes   
4   59  blue-collar  married  secondary      no        0     yes   no   

    contact  day month  duration  campaign  pdays  previous poutcome   y  
0  cellular   19   oct        79         1     -1         0  unknown  no  
1  cellular   11   may       220         1    339         4  failure  no  
2  cellular   16   apr       185         1    330         1  failure  no  
3   unknown    3   jun       199         4     -1         0  un

In [24]:
# Separate input features and target variable
X = df.drop("y", axis=1)
y = df["y"]

print("Dataset loaded successfully.")
print("X shape:", X.shape)
print("y shape:", y.shape)

Dataset loaded successfully.
X shape: (4521, 16)
y shape: (4521,)


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns

print("Numerical features:", list(numeric_features))
print("Categorical features:", list(categorical_features))

# Preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("\nPreprocessing setup completed.")

Numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Preprocessing setup completed.


In [26]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training data shape: (3616, 16)
Testing data shape: (905, 16)
Training target shape: (3616,)
Testing target shape: (905,)


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.base import clone

# Define the five classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42)
}

# Train the models
trained_models = {}

for model_name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[model_name] = pipeline

    print(f"{model_name} trained successfully!")

print("\nAll five models have been trained.")

Logistic Regression trained successfully!
Decision Tree trained successfully!
KNN trained successfully!
Gaussian Naive Bayes trained successfully!


/home/s2026ak05463/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM trained successfully!

All five models have been trained.


In [28]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

import pandas as pd

results = []

for model_name, model in trained_models.items():

    # Predict test data
    y_pred = model.predict(X_test)

    # Probability for the positive class: "yes"
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, pos_label="yes", zero_division=0)
    recall = recall_score(y_test, y_pred, pos_label="yes", zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label="yes", zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

# Create comparison table
results_df = pd.DataFrame(results)

# Display results
display(results_df.round(4))

,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8928,0.8913,0.5636,0.2981,0.3899,0.3579
1,Decision Tree,0.8586,0.6942,0.4032,0.4808,0.4386,0.3602
2,KNN,0.8840,0.7692,0.4872,0.1827,0.2657,0.2477
3,Gaussian Naive Bayes,0.8210,0.7881,0.3092,0.4519,0.3672,0.2737
4,SVM,0.8939,0.8830,0.5909,0.2500,0.3514,0.3374


In [29]:
import os
import joblib

# Create a folder to store trained models
os.makedirs("models", exist_ok=True)

# Save each trained model
for model_name, model in trained_models.items():

    filename = model_name.replace(" ", "_").lower() + ".pkl"

    joblib.dump(
        model,
        os.path.join("models", filename)
    )

    print(f"{model_name} saved successfully!")

print("\nAll models saved in the models folder.")

Logistic Regression saved successfully!
Decision Tree saved successfully!
KNN saved successfully!
Gaussian Naive Bayes saved successfully!
SVM saved successfully!

All models saved in the models folder.


In [30]:
# Combine test features and actual target values
test_data = X_test.copy()
test_data["y"] = y_test

# Save the test dataset
test_data.to_csv("test_data.csv", index=False)

print("Test data saved successfully!")
print("Test data shape:", test_data.shape)

display(test_data.head())

Test data saved successfully!
Test data shape: (905, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
2032,59,management,married,primary,no,1727,no,no,telephone,25,may,500,4,358,2,failure,yes
943,40,technician,married,secondary,no,109,no,no,unknown,20,jun,402,22,-1,0,unknown,no
408,44,management,married,secondary,no,438,yes,no,telephone,8,may,37,3,303,1,failure,no
3034,29,services,married,primary,no,-102,no,no,unknown,4,jul,134,1,-1,0,unknown,no
3447,53,management,married,tertiary,no,146,no,no,unknown,30,may,150,1,-1,0,unknown,no
